In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install tokenizers datasets mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 733.8/733.8 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 20.0 MB/s eta 0:00:00


In [ ]:
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import os
import mlflow
import pandas as pd
from datasets import load_dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
from tokenizers.processors import TemplateProcessing
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

In [ ]:
# === Config ===
VOCAB_SIZE = 32000
OUTPUT_DIR = "/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2"
SPECIAL_TOKENS = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]", "<s>", "</s>"]

In [ ]:
from datasets import load_dataset

ds = load_dataset("EmanKhater/Tashkeela", split="train")
tashkeela_texts = ds["text"]
print(f"✅ Tashkeela samples: {len(tashkeela_texts)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Tashkeela.zip:   0%|          | 0.00/138M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3090000 [00:00<?, ? examples/s]

✅ Tashkeela samples: 3090000


In [ ]:
egyptian = load_dataset("MBZUAI-Paris/Egyptian-SFT-Mixture", split="train")
egyptian_texts = egyptian["messages"]

In [ ]:
print(f"✅ Egyptian-SFT-Mixture samples: {len(egyptian)}")

✅ Egyptian-SFT-Mixture samples: 1817288


In [ ]:
import kagglehub
import pandas as pd
import os

# ✅ Download dataset from Kaggle
arsarcasm_path = kagglehub.dataset_download("abraralotaibi00/arsarcasm-v2")
print("📦 Dataset downloaded to:", arsarcasm_path)

📦 Dataset downloaded to: /kaggle/input/arsarcasm-v2


In [ ]:
# ✅ Locate the correct file — it's usually in .tsv or .csv
csv_candidates = [
    os.path.join(arsarcasm_path, f)
    for f in os.listdir(arsarcasm_path)
    if f.endswith(".csv") or f.endswith(".tsv")
]

In [ ]:
# Combine both sources
all_texts = tashkeela_texts + egyptian_texts
print(f"✅ Total combined lines: {len(all_texts)}")

✅ Total combined lines: 4907288


In [ ]:
# # === Add to your all_texts ===
# all_texts += texts
# print("📊 Total lines after adding ArSarcasm:", len(all_texts))

📊 Total lines after adding ArSarcasm: 4910288


In [ ]:
import re
from tqdm import tqdm

# Arabic letters + numerals + basic punctuation
arabic_char_pattern = re.compile(r'[\u0600-\u06FF\d\s.,!?،؛:()\[\]{}«»<>…\"\'-]+')

def is_valid_utf8(text):
    try:
        text.encode("utf-8").decode("utf-8")
        return True
    except UnicodeDecodeError:
        return False

def is_mostly_arabic(text, threshold=0.6):
    arabic_chars = re.findall(r'[\u0600-\u06FF]', text)
    return len(arabic_chars) / max(len(text), 1) >= threshold

def is_clean_text(text):
    if not text.strip(): return False
    if len(text.split()) < 5: return False
    if not is_valid_utf8(text): return False
    if not is_mostly_arabic(text): return False
    if len(set(text)) < 10: return False
    return True

# 🧹 Apply filters
filtered_texts = [
    t.strip()
    for t in tqdm(all_texts)
    if isinstance(t, str) and is_clean_text(t.strip())
]

print(f"✅ Filtered clean lines: {len(filtered_texts):,}")


100%|██████████| 4907288/4907288 [01:11<00:00, 68738.79it/s]  

✅ Filtered clean lines: 2,637,241


In [ ]:
def is_minimally_clean(text):
    if not text.strip():
        return False
    if not is_valid_utf8(text):
        return False
    arabic_ratio = sum(1 for c in text if '\u0600' <= c <= '\u06FF') / len(text)
    return arabic_ratio > 0.2

    # 🧹 Apply filters
clean_texts = [
    t.strip()
    for t in tqdm(all_texts)
    if isinstance(t, str) and is_clean_text(t.strip())
]

print(f"✅ Filtered clean lines: {len(clean_texts):,}")

In [ ]:
with open("/content/drive/MyDrive/filtered_adaptive_corpus.txt", "w", encoding="utf-8") as f:
    for line in filtered_texts:
        f.write(line.strip() + "\n")

In [ ]:
# 🧹 Apply filters
texts = [
    t.strip()
    for t in tqdm(all_texts)
    if isinstance(t, str) and is_clean_text(t.strip())
]

print(f"✅ Filtered clean lines: {len(texts):,}")

100%|██████████| 4907288/4907288 [01:06<00:00, 73349.73it/s]  

✅ Filtered clean lines: 2,637,241


In [ ]:
# === 2. Train the Tokenizer ===
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))

from tokenizers.pre_tokenizers import Sequence, Whitespace, Punctuation

tokenizer.pre_tokenizer = Sequence([
    Whitespace(),
    Punctuation()
])

trainer = trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=SPECIAL_TOKENS
)

In [ ]:
def is_valid_line(text):
    return isinstance(text, str) and len(text.strip()) > 10 and is_valid_utf8(text)

all_texts = [sample for sample in all_texts if is_valid_line(sample)]

In [ ]:
tokenizer.train_from_iterator(all_texts, trainer=trainer)

In [ ]:
# === 3. Postprocessing & Saving ===
from tokenizers.processors import TemplateProcessing
tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> </s> $B </s>",
    special_tokens=[
        ("<s>", tokenizer.token_to_id("<s>")),
        ("</s>", tokenizer.token_to_id("</s>")),
    ]
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
tokenizer_path = os.path.join(OUTPUT_DIR, "tokenizer.json")
tokenizer.save(tokenizer_path)
print("✅ Tokenizer saved to:", tokenizer_path)

✅ Tokenizer saved to: /content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2/tokenizer.json


In [ ]:
# === 4. Wrap in Hugging Face Format ===
hf_tokenizer = PreTrainedTokenizerFast(tokenizer_file=tokenizer_path)
hf_tokenizer.add_special_tokens({
    "pad_token": "[PAD]",
    "unk_token": "[UNK]",
    "cls_token": "[CLS]",
    "sep_token": "[SEP]",
    "mask_token": "[MASK]",
    "bos_token": "<s>",
    "eos_token": "</s>"
})
hf_tokenizer.save_pretrained(OUTPUT_DIR)

('/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2/tokenizer_config.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2/special_tokens_map.json',
 '/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2/tokenizer.json')

In [ ]:
from transformers import PreTrainedTokenizerFast

# Wrap the trained tokenizer JSON
tokenizer = PreTrainedTokenizerFast(tokenizer_file="/content/drive/MyDrive/FINAL_GRAD_PROJ/arabic_tokenizer_v2/tokenizer.json")

# Add special tokens explicitly (if not already saved)
tokenizer.add_special_tokens({
    "pad_token": "[PAD]",
    "unk_token": "[UNK]",
    "cls_token": "[CLS]",
    "sep_token": "[SEP]",
    "mask_token": "[MASK]",
    "bos_token": "<s>",
    "eos_token": "</s>"
})

0

In [ ]:
sample = "ما هو الذكاء الاصطناعي؟"
tokens = tokenizer.encode(sample)
print("🔤 Original:", sample)
print("🧩 Tokens:", tokenizer.tokenize(sample))
print("🧱 Token IDs:", tokens)
print("🔁 Decoded:", tokenizer.decode(tokens))
print(f"❗ UNK Count: {tokens.count(tokenizer.unk_token_id)}")

🔤 Original: ما هو الذكاء الاصطناعي؟
🧩 Tokens: ['م', 'ا', 'ه', 'و', 'الذ', 'ك', 'ا', 'ء', 'ال', 'ا', 'ص', 'ط', 'ن', 'ا', 'ع', 'ي', '؟']
🧱 Token IDs: [5, 122, 98, 124, 125, 13016, 120, 98, 92, 151, 98, 112, 114, 123, 98, 116, 127, 91, 6]
🔁 Decoded: <s> م ا ه و الذ ك ا ء ال ا ص ط ن ا ع ي ؟ </s>
❗ UNK Count: 0


In [ ]:
from statistics import mean

# Load 100 random samples for quick evaluation (you can increase this)
test_samples = all_texts[:100]

token_lengths = []
unique_token_counts = []

for sample in test_samples:
    token_ids = tokenizer.encode(sample)
    token_lengths.append(len(token_ids))
    unique_token_counts.append(len(set(token_ids)))

avg_tokens = mean(token_lengths)
avg_unique_tokens = mean(unique_token_counts)

print(f"✅ Average tokens per sample: {avg_tokens:.2f}")
print(f"✅ Average unique tokens per sample: {avg_unique_tokens:.2f}")
print(f"🔍 Max tokens: {max(token_lengths)}, Min tokens: {min(token_lengths)}")


✅ Average tokens per sample: 15.99
✅ Average unique tokens per sample: 15.27
🔍 Max tokens: 61, Min tokens: 5


In [ ]:
print(all_texts[0])
print(repr(all_texts[0]))  # show hidden characters

وَالمَحْجُومُ } فَأَجَابُوا عَنْهُ بِأَنَّهُ مَنْسُوخٌ بِخَبَرِ البُخَارِيِّ { أَنَّهُ صَلَّى اللَّهُ عَلَيْهِ وَسَلَّمَ احْتَجَمَ ،
'وَالمَحْجُومُ } فَأَجَابُوا عَنْهُ بِأَنَّهُ مَنْسُوخٌ بِخَبَرِ البُخَارِيِّ { أَنَّهُ صَلَّى اللَّهُ عَلَيْهِ وَسَلَّمَ احْتَجَمَ ،'
